# Read a document with Fabric-managed Content Understanding

This example uses Fabric's runtime endpoint and identity. No Azure resource endpoint, API key, or Content Understanding SDK is needed. It generates a synthetic PDF, saves its operation handle, and resumes polling without submitting the same document again.

**Prerequisites:** a Fabric Spark notebook with an attached lakehouse, matching SynapseML Python and JVM artifacts containing this integration, and Fabric-managed Content Understanding enabled for the capacity and region. See the [SynapseML installation guide](https://microsoft.github.io/SynapseML/docs/Get%20Started/Install%20SynapseML/).

`prebuilt-read` demonstrates OCR, not LLM-based field extraction. Managed analyzer/model availability is controlled by Fabric. For LLM-backed invoice fields or custom analyzers, use the [BYO Foundry invoice notebook](https://github.com/microsoft/SynapseML/blob/master/docs/Explore%20Algorithms/AI%20Services/Quickstart%20-%20Content%20Understanding.ipynb). The managed route does not support analyzer administration.

Analysis is billable. This example does not provision resources, change model defaults, or enable service features.

In [ ]:
from synapse.ml.services.contentunderstanding import ContentUnderstanding

journal_path = "Files/content-understanding/fabric-read-demo-v1"
analyzer = (
    ContentUnderstanding()
    .setAnalyzerId("prebuilt-read")
    .setDocumentBytesCol("body")
    .setDocumentName("synthetic-document.pdf")
    .setMimeType("application/pdf")
    .setPollingIntervalSeconds(6)
    .setMaxPollAttempts(120)
)
# Do not set an endpoint or credentials for this managed example.

## Create a deterministic PDF

This document contains no customer data and requires no document-generation package.

In [ ]:
def example_pdf():
    text = b"BT /F1 16 Tf 50 730 Td (SynapseML Content Understanding example) Tj ET"
    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [4 0 R] /Count 1 >>",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
        b"/Resources << /Font << /F1 3 0 R >> >> /Contents 5 0 R >>",
        f"<< /Length {len(text)} >>\nstream\n".encode("ascii") + text + b"\nendstream",
    ]
    data = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for number, obj in enumerate(objects, 1):
        offsets.append(len(data))
        data.extend(f"{number} 0 obj\n".encode("ascii") + obj + b"\nendobj\n")
    xref = len(data)
    data.extend(f"xref\n0 {len(offsets)}\n0000000000 65535 f \n".encode("ascii"))
    for offset in offsets[1:]:
        data.extend(f"{offset:010d} 00000 n \n".encode("ascii"))
    data.extend(
        f"trailer\n<< /Size {len(offsets)} /Root 1 0 R >>\n"
        f"startxref\n{xref}\n%%EOF\n".encode("ascii")
    )
    return data


documents = spark.createDataFrame(
    [("synthetic-document-v1", example_pdf())], "documentId string, body binary"
)

## Save the handle and await the result

Only one writer may own this journal. Keep the same document, ID, configuration, and journal path when resuming. `submit` saves the accepted operation handle before polling. A crash between service acceptance and saving the handle can still cause a duplicate submission.

The service processes jobs asynchronously. SynapseML polls during the call; it does not leave a detached polling daemon running. HTTP 429 responses retry indefinitely by default. JVM interruption stops retries; a notebook interrupt that only stops Python waiting may not stop the driver call. Set `retryOnThrottling` to `False` when bounded attempts are required.

In [ ]:
analyzer.setOperationMode("submit").writeToPath(
    documents, "documentId", journal_path, format="delta"
)
analyzer.setOperationMode("analyze").writeToPath(
    documents, "documentId", journal_path, format="delta"
)
saved = analyzer.readPath(spark, journal_path, "delta")
saved.select("documentId", "status", "httpStatus", "error").show(truncate=False)
assert saved.where("status = 'Succeeded'").count() == 1, (
    "Inspect the service error. For pending work, resume with the same journal; "
    "do not treat a running or failed operation as an empty result."
)
saved.selectExpr(
    "documentId",
    "get_json_object(rawResponse, '$.result.contents[0].markdown') AS text",
).show(truncate=False)

The last query reads saved results and does not call the service. Repeating the writer cells skips already completed documents. Resume pending operations promptly because service results have limited retention.

For an ordinary Spark pipeline, use `analyzer.setConcurrency(4).transform(documents)`. Concurrency applies within each partition; the durable writers above are sequential. Persist a transform's results before displaying them because repeating Spark actions can repeat billable submissions.

See [Content Understanding parameters and recovery](https://microsoft.github.io/SynapseML/docs/Explore%20Algorithms/AI%20Services/Content%20Understanding/) and the [REST API reference](https://learn.microsoft.com/rest/api/contentunderstanding/content-analyzers/analyze?view=rest-contentunderstanding-2025-11-01).